# Tick data

For optimum results this notebook should be run during the Forex trading session.

In [1]:
# to import local code
# https://stackoverflow.com/questions/61058798/python-relative-import-in-jupyter-notebook
# https://stackoverflow.com/questions/34478398/import-local-function-from-a-module-housed-in-another-directory-with-relative-im

import os, sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    # sys.path.append(parent_dir)
    sys.path.insert(0, parent_dir)
    print(parent_dir)


c:\Users\Jimmy\source\erdewit\ib_insync


In [2]:
from ib_insync import *
util.startLoop()


In [3]:
ib = IB()
host_1 = '127.0.0.1'
host_2 = '192.168.1.90'
port_1 = 7496
port_2 = 7497
port_3 = 4002
ib.connect(host_1, port_2, clientId=15)

<IB connected to 127.0.0.1:7497 clientId=15>

In [4]:
import logging

In [5]:
# util.logToConsole(logging.DEBUG) # show regular traffic

contracts = [Stock(x, 'SMART', 'USD') for x in ['AAPL','NVDA','TSLA','META','SPY','QQQ']]
ib.qualifyContracts(*contracts)
for contract in contracts:
    ib.reqMktData(contract, '', False, False)
# ib.sleep(3)
# ib.disconnect()


### Streaming tick data

Create some Forex contracts:

In [6]:
contracts = [Forex(pair) for pair in ('EURUSD', 'USDJPY', 'GBPUSD', 'USDCHF', 'USDCAD', 'AUDUSD')]
ib.qualifyContracts(*contracts)

eurusd = contracts[0]

Request streaming ticks for them:

In [7]:
for contract in contracts:
    ib.reqMktData(contract, '', False, False)

In [8]:
t=ib.tickers()[4]

In [9]:
import datetime

In [10]:
(t.time - datetime.datetime.now(tz=datetime.timezone.utc)).total_seconds()

-0.638227

In [11]:
t.time, t.marketDataType, t.bid, t.ask, t.close, t.volume, t.open, t.high, t.low, t.contract.localSymbol, t.midpoint()

AttributeError: 'Ticker' object has no attribute 'open'

In [12]:
ib.reqMarketDataType(1)

In [13]:
def onPendingTickers(tickers):
    for t in tickers:
        # print(t.bidSize, t.bid, t.ask, t.askSize, t.high, t.low, t.close, t.contract.localSymbol)
        print(t.bid, t.ask, t.last, t.contract.localSymbol, t.marketDataType)

In [14]:
ib.pendingTickersEvent += onPendingTickers
ib.sleep(10)
ib.pendingTickersEvent -= onPendingTickers

0.65548 0.6555 nan AUD.USD 1
1.36177 1.36179 nan USD.CAD 1
1.17463 1.17465 nan EUR.USD 1
146.38 146.383 nan USD.JPY 1
1.35164 1.35167 nan GBP.USD 1
0.79275 0.79279 nan USD.CHF 1
331.31 331.36 331.34 TSLA 1
331.31 331.36 331.3 TSLA 1
561.09 561.1 561.1 QQQ 1
561.09 561.1 561.1 QQQ 1
627.82 627.83 627.83 SPY 1
627.82 627.83 627.82 SPY 1
627.82 627.83 627.82 SPY 1
168.33 168.34 168.34 NVDA 1
561.08 561.09 561.1 QQQ 1
627.82 627.83 627.82 SPY 1
213.03 213.04 213.03 AAPL 1
331.27 331.36 331.3 TSLA 1
146.377 146.38 nan USD.JPY 1
1.36175 1.36177 nan USD.CAD 1
1.17465 1.17467 nan EUR.USD 1
0.65549 0.6555 nan AUD.USD 1
0.79274 0.79278 nan USD.CHF 1
1.35165 1.3517 nan GBP.USD 1
0.65549 0.6555 nan AUD.USD 1
0.79274 0.79278 nan USD.CHF 1
1.17465 1.17466 nan EUR.USD 1
0.65552 0.65555 nan AUD.USD 1
146.377 146.381 nan USD.JPY 1
1.36174 1.36177 nan USD.CAD 1
0.65552 0.65554 nan AUD.USD 1
0.79273 0.79278 nan USD.CHF 1
1.17464 1.17465 nan EUR.USD 1
1.35165 1.3517 nan GBP.USD 1
146.377 146.38 nan USD.JP

Wait a few seconds for the tickers to get filled.

In [15]:
ticker = ib.ticker(eurusd)
ib.sleep(2)

ticker

Ticker(contract=Forex('EURUSD', conId=12087792, exchange='IDEALPRO', localSymbol='EUR.USD', tradingClass='EUR.USD'), time=datetime.datetime(2025, 7, 22, 15, 57, 3, 579080, tzinfo=datetime.timezone.utc), minTick=1e-05, bid=1.17459, bidSize=2175000.0, ask=1.17461, askSize=1000000.0, prevBid=1.1746, prevBidSize=3175000.0, prevAsk=1.17465, prevAskSize=2000000.0, volume=0.0, high=1.17475, low=1.1679, close=1.1695)

The price of Forex ticks is always nan. To get a midpoint price use ``midpoint()`` or ``marketPrice()``.

The tickers are kept live updated, try this a few times to see if the price changes:

In [32]:
contracts[0].pair()

'EURUSD'

In [19]:
ticker.marketPrice()

1.1745549999999998

The following cell will start a 30 second loop that prints a live updated ticker table.
It is updated on every ticker change.

In [36]:
from IPython.display import display, clear_output
import pandas as pd

df = pd.DataFrame(
    index=[c.conId for c in contracts],
    columns=['symbol','bidSize', 'bid', 'ask', 'askSize', 'high', 'low', 'close', 'time'])

def onPendingTickers(tickers):
    for t in tickers:
        df.loc[t.contract.conId] = (
            t.contract.symbol, t.bidSize, t.bid, t.ask, t.askSize, t.high, t.low, t.close, t.time)
        clear_output(wait=True)
    display(df)        

ib.pendingTickersEvent += onPendingTickers
ib.sleep(60)
ib.pendingTickersEvent -= onPendingTickers

,symbol,bidSize,bid,ask,askSize,high,low,close,time
12087792,EUR,5000000.0,1.17528,1.17532,1000000.0,1.17605,1.16790,1.16950,2025-07-22 20:56:27.885501+00:00
15016059,USD,1000000.0,146.67000,146.68000,1000000.0,147.94500,146.31000,147.36000,2025-07-22 20:56:21.478707+00:00
12087797,GBP,1000000.0,1.35321,1.35331,1000000.0,1.35330,1.34620,1.34920,2025-07-22 20:56:27.764474+00:00
12087820,USD,1000000.0,0.79216,0.79222,2000000.0,0.79920,0.79205,0.79800,2025-07-22 20:56:27.965505+00:00
15016062,USD,1000000.0,1.36047,1.36052,2000000.0,1.36955,1.36030,1.36820,2025-07-22 20:56:27.364042+00:00
14433401,AUD,5000000.0,0.65553,0.65557,3000000.0,0.65580,0.65040,0.65245,2025-07-22 20:56:21.868781+00:00
4815747,NVDA,100.0,166.46000,166.49000,400.0,171.39000,164.58000,171.38000,2025-07-22 20:56:26.609677+00:00
76792991,TSLA,100.0,331.56000,331.60000,300.0,335.41000,321.55000,328.49000,2025-07-22 20:56:25.846522+00:00
320227571,QQQ,1700.0,560.59000,560.60000,100.0,564.35000,558.61000,564.17000,2025-07-22 20:56:28.099862+00:00
756733,SPY,2800.0,628.93000,628.97000,7300.0,629.73000,626.19000,628.77000,2025-07-22 20:56:28.111862+00:00


New tick data is available in the 'ticks' attribute of the pending tickers.
The tick data will be cleared before the next update.

To stop the live tick subscriptions:

In [12]:
for contract in contracts:
    ib.cancelMktData(contract)

priceSizeTick: Unknown reqId: 25
tickSize: Unknown reqId: 25
tickSize: Unknown reqId: 25
tickSize: Unknown reqId: 26
tickSize: Unknown reqId: 26


### Tick by Tick data ###

The ticks in the previous section are time-sampled by IB in order to cut on bandwidth. So with ``reqMktdData`` not every tick from the exchanges is sent. The promise of ``reqTickByTickData`` is to send every tick, just how it appears in the TWS Time & Sales window. This functionality is severly nerfed by a total of just three simultaneous subscriptions, where bid-ask ticks and sale ticks also use up a subscription each.

The tick-by-tick updates are available from ``ticker.tickByTicks`` and are signalled by ``ib.pendingTickersEvent`` or ``ticker.updateEvent``.

In [ ]:
ticker = ib.reqTickByTickData(eurusd, 'BidAsk')
ib.sleep(2)
print(ticker)

ib.cancelTickByTickData(ticker.contract, 'BidAsk')

### Historical tick data

Historical tick data can be fetched with a maximum of 1000 ticks at a time. Either the start time or the end time must be given, and one of them must remain empty:

In [ ]:
import datetime

start = ''
end = datetime.datetime.now()
ticks = ib.reqHistoricalTicks(eurusd, start, end, 1000, 'BID_ASK', useRth=False)

ticks[-1]

In [1]:
ib.disconnect()

NameError: name 'ib' is not defined